# 公司员工数据集生成器

## 练习目标（理念）

用 **Hugging Face Transformers** 加载量化后的本地/云端大模型（这里是 Llama），根据用户描述的公司背景，**生成一份虚构的员工 CSV 数据集**，再用 **Gradio** 做成可交互的小工具。

## 和本课 Week 3 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Hugging Face Hub 登录 | `login(hf_token, ...)` |
| 因果语言模型 Causal LM | `AutoModelForCausalLM` |
| 分词器 Tokenizer | `AutoTokenizer` + chat template |
| 4-bit 量化 Quantization | `BitsAndBytesConfig` |
| Chat messages（system / user） | system 定「怎么生成数据」，user 放公司简介 |
| Gradio 界面 | `gr.Blocks` / `Dataframe` / `Textbox` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `HF_TOKEN`（Hugging Face 访问令牌）
3. 机器需有可用 **CUDA**（代码里用 `assert torch.cuda.is_available()` 检查）
4. 最后一格会启动 Gradio：在文本框里描述公司，点 Generate 生成表格


### 我们的助理回来了

本练习会生成 **CSV 格式** 的虚构公司员工数据集（姓名、部门、职位、资历、薪资与奖金等），供后续分析或演示使用。


In [ ]:
# ========== 安装依赖：首次运行或环境缺包时执行 ==========

# 升级安装 bitsandbytes：4-bit 量化（Quantization）所需；版本约束 >=0.46.1
!pip install -U bitsandbytes>=0.46.1
# 安装 accelerate：Transformers 多卡/自动 device_map 常用配套库
!pip install accelerate
# 安装 hf_xet：Hugging Face 下载/传输相关辅助包
!pip install hf_xet


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# IMPORTS（取消注释以在 google collab 中使用）
# 从 google.colab 导入用户数据
# 从 huggingface_hub 导入 login：用 HF Token 登录 Hugging Face Hub，才能拉受控模型
from huggingface_hub import login
# 从 transformers 导入：分词器、因果语言模型、BitsAndBytes 量化配置
from transformers import AutoTokenizer, AutoModelForCausalLM,  BitsAndBytesConfig
# 导入 PyTorch：张量运算与 CUDA 设备检查
import torch
# 导入 Gradio：快速搭 Web 交互界面（Textbox / Dataframe / Button）
import gradio as gr
# 导入 pandas：把模型吐出的 CSV 文本解析成 DataFrame 表格
import pandas as pd
# 导入标准库 os：读环境变量（Environment Variables），例如 HF_TOKEN
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 导入标准库 io：用 StringIO 把字符串当成「文件」交给 pandas.read_csv
import io
# 导入标准库 re：用正则从模型回复里抠出 ```csv ... ``` 代码块
import re


In [ ]:
# ========== 登录 Hugging Face + 检查 CUDA ==========

# 登录拥抱脸（Hugging Face Hub）
# 若在 Google Colab，可改用下一行从 userdata 取 HF_TOKEN（当前保持注释）
# hf_token = userdata.get('HF_TOKEN')
# 加载 .env：override=True 表示用文件值覆盖已有环境变量
load_dotenv(override=True)
# 从环境变量读取 Hugging Face 访问令牌（不要把真 token 写进笔记本）
hf_token = os.getenv('HF_TOKEN')
# 登录 Hub；add_to_git_credential=True 便于后续 git/lfs 凭据复用
login(hf_token, add_to_git_credential=True)
# 检查 CUDA 是否可用：本练习假定用 GPU 跑量化 Llama；不可用则立刻断言失败
assert torch.cuda.is_available(), "CUDA is unavailable"
# 设备名字符串：后面生成时张量要放到 cuda
device = "cuda"


In [ ]:
# ========== 模型 ID：集中写在一处，后面只改这里 ==========

# 我们生成数据集的模型：Llama 3.1 8B Instruct（指令微调版，适合聊天式生成）
# 字符串必须与 Hugging Face 上的模型仓库名一致，改译/改写会拉错模型
LLAMA = 'meta-llama/Llama-3.1-8B-Instruct'


In [ ]:
# ========== System Prompt：告诉模型「如何生成员工 CSV」==========

# 提示（prompt）：发给模型的系统角色指令；正文必须保持英文原样，改译会改变模型行为
system_prompt = """
„You generate an artificial dataset based on company info provided by the user.

Dataset requirements:

Features: First_name, Last_name, department, position, seniority (Junior, Mid, Senior), basic_salary (numeric), bonus (numeric).

Rules: Respond just with dataset, don't reply like "Here is your dataset: ... " . No NULL values. Data must be realistic: 'Senior' must earn more than 'Junior' within the same department. Salaries should reflect industry standards for the given department.

Format: CSV format inside a code block.

Quantity: Generate exactly 10 rows of data.

Example:

Question:
About Allegro
For around a quarter of a century, we have been serving consumers and promoting the idea of entrepreneurship in one of the most innovative areas of the economy. We are the go-to online shopping destination for millions of consumers and the biggest e-commerce player of European origin, creating the place to do business for thousands of companies, most of them small and medium enterprises. We create innovations that improve the daily lives of millions of Europeans, allowing them to shop for products they need while saving money and time.

Response:
First_name,Last_name,department,basic_salary,bonus,position,seniority
Krzysztof,Kowalski,Sales,40000,10000,Sales Manager,Senior
Aleksandra,Kaczmarek,Logistics,35000,8000,Logistics Coordinator,Mid
Tomasz,Wójcik,Marketing,45000,12000,Marketing Director,Senior
Anna,Kwiatkowska,HR,30000,6000,HR Specialist,Junior
Michał,Zieliński,IT,55000,15000,IT Manager,Senior
Ewa,Sadowska,Finance,38000,10000,Financial Analyst,Mid
Mariusz,Kosowski,Sales,42000,11000,Sales Representative,Senior
Paulina,Majchrzak,Logistics,32000,7000,Logistics Assistant,Junior
Piotr,Bednarczyk,Marketing,48000,13000,Marketing Specialist,Senior
Julia,Gąsior,HR,29000,5000,HR Assistant,Junior
"""


In [ ]:
# ========== 4-bit 量化配置：省显存地加载 8B 模型 ==========

# BitsAndBytesConfig：告诉 Transformers 如何用 bitsandbytes 做量化加载
quant_config = BitsAndBytesConfig(
    # load_in_4bit：权重以 4-bit 形式加载，显著降低显存占用
    load_in_4bit=True,
    # 双重量化（double quant）：对量化常数再量化一层，进一步省显存
    bnb_4bit_use_double_quant=True,
    # 计算时用 bfloat16：在支持的 GPU 上稳定且省带宽
    bnb_4bit_compute_dtype=torch.bfloat16,
    # 量化类型 nf4：NormalFloat4，对神经网络权重分布更友好的常见选择
    bnb_4bit_quant_type="nf4"
)


In [ ]:
# ========== 加载模型 + generate：聊天生成并解析成 DataFrame ==========

# 打印进度：大模型下载/加载可能较久
print("Loading model...")
# 按模型 ID 加载分词器（Tokenizer）：负责文本 ↔ token id
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
# 因果 LM 常缺独立 pad_token：用 eos 充当 padding，避免 generate 告警/报错
tokenizer.pad_token = tokenizer.eos_token
# 加载因果语言模型；device_map="auto" 自动把层放到可用设备；套用上面的 4-bit 量化配置
model = AutoModelForCausalLM.from_pretrained(LLAMA,  device_map="auto", quantization_config=quant_config)

def generate(messages, max_new_tokens=512):
    # messages：OpenAI 风格的 [{"role","content"}, ...]；apply_chat_template 按 Llama chat 模板拼输入
    # return_tensors="pt" 得到 PyTorch 张量；add_generation_prompt=True 在末尾加上「助手开始回答」标记
    # .to("cuda")：把输入放到 GPU
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

    # attention_mask：全 1 表示这些位置都是有效 token（无 padding 时这样即可）
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")

    # model.generate：自回归续写；只生成最多 max_new_tokens 个新 token，遇到 eos 可提前停
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    )

    # 解码：只取「新生成」那一段（切片掉输入长度），并跳过特殊 token
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)

    # 用正则从回复里提取 ``` 或 ```csv 代码块中的 CSV 正文；DOTALL 让 . 能匹配换行
    csv_match = re.search(r"```(?:csv)?\n(.*?)\n```", response, re.DOTALL)
    # 匹配成功取捕获组；否则退回整段 response（容错）
    csv_text = csv_match.group(1) if csv_match else response

    try:
        # StringIO：把字符串当成文件流；read_csv 解析成 DataFrame
        df = pd.read_csv(io.StringIO(csv_text.strip()))
        return df
    except Exception as e:
        # 解析失败时返回错误说明表，便于在 Gradio 里看见原因（错误文案保持英文原样）
        return pd.DataFrame({"Error": [f"Could not parse CSV: {str(e)}"], "Raw Output": [response[:100]]})


In [ ]:
# ========== Gradio 界面：输入公司简介 → 展示生成的员工表 ==========

# 广播用户界面（Gradio UI）
def handle_interaction(message):
    # 组装聊天 messages：system 固定为上面的生成规则；user 是用户输入的公司描述
    messages = [{"role":"system", "content":system_prompt},{"role":"user", "content":message}]
    # 调用 generate：返回 DataFrame，交给 Gradio Dataframe 组件显示
    return generate(messages)

# gr.Blocks：自定义布局的 Gradio 应用上下文
with gr.Blocks() as ui:
    # 标题 Markdown（界面文案保持英文原样，避免改交互文案）
    gr.Markdown("### AI Company Dataset Generator")
    # 第一行：输出表格组件
    with gr.Row():
        output_df = gr.Dataframe(label="Generated Dataset")
    # 第二行：输入框 + 生成按钮
    with gr.Row():
        message = gr.Textbox(label="Enter details about the company (e.g. 'Software house in Poland'):")
        btn = gr.Button("Generate")

    # 在输入框按 Enter：触发 handle_interaction，结果写入 output_df
    message.submit(handle_interaction, inputs=[message], outputs=[output_df])
    # 点击按钮：同样绑定到 handle_interaction
    btn.click(handle_interaction, inputs=[message], outputs=[output_df])

# 启动 Gradio：debug=True 便于看报错；auth 简单账号密码；inbrowser=True 尝试自动打开浏览器
ui.launch(debug=True, auth=('eryk', 'eryk'), inbrowser=True)
